In [1]:
import os
from pathlib import Path
import numpy as np
import SimpleITK as sitk
import pydicom
import matplotlib.pyplot as plt
#import pydicom_seg as dcmseg
from radiomics import featureextractor
import pandas as pd
import logging
# suppress verbose pyradiomics/radiomics informational messages
logging.getLogger('radiomics').setLevel(logging.ERROR)
logging.getLogger('radiomics.featureextractor').setLevel(logging.ERROR)
logging.getLogger('pyradiomics').setLevel(logging.ERROR)

In [2]:
full_radiomics = pd.read_csv(os.path.expanduser('~/project/xAI-in-NSCLC/FULL_radiomics_features_per_slice.csv'))

In [3]:
full_radiomics.head()

,PatientID,slice_no,original_shape2D_Elongation,original_shape2D_MajorAxisLength,original_shape2D_MaximumDiameter,original_shape2D_MeshSurface,original_shape2D_MinorAxisLength,original_shape2D_Perimeter,original_shape2D_PerimeterSurfaceRatio,original_shape2D_PixelSurface,...,wavelet-L_gldm_LargeDependenceLowGrayLevelEmphasis,wavelet-L_gldm_LowGrayLevelEmphasis,wavelet-L_gldm_SmallDependenceEmphasis,wavelet-L_gldm_SmallDependenceHighGrayLevelEmphasis,wavelet-L_gldm_SmallDependenceLowGrayLevelEmphasis,wavelet-L_ngtdm_Busyness,wavelet-L_ngtdm_Coarseness,wavelet-L_ngtdm_Complexity,wavelet-L_ngtdm_Contrast,wavelet-L_ngtdm_Strength
0,LUNG1-001,65,0.544937,13.659473,13.566840,78.678131,7.443553,35.768962,0.454624,79.154968,...,0.012320,0.012319,0.981928,121004.765060,0.012319,0.001196,0.045945,232302.606827,188.887364,4646.827408
1,LUNG1-001,66,0.513136,25.130245,25.633603,248.432159,12.895239,66.822663,0.268978,248.908997,...,0.004195,0.003963,0.925287,146055.210728,0.003905,0.000540,0.035563,247872.636083,20.744195,2395.297089
2,LUNG1-001,67,0.455774,51.188040,50.706074,826.358795,23.330172,137.649725,0.166574,826.835632,...,0.001178,0.001177,0.948802,117398.479239,0.001176,0.000529,0.017956,322249.286049,2.527396,1310.173794
3,LUNG1-001,68,0.661752,64.091619,68.589178,2030.849457,42.412765,197.330095,0.097166,2031.326294,...,0.000826,0.000791,0.963954,131113.500000,0.000782,0.000399,0.012122,963479.546047,1.742460,1705.476464
4,LUNG1-001,69,0.606515,76.090875,81.967283,2584.934235,46.150227,232.290046,0.089863,2585.411072,...,0.000385,0.000384,0.953871,184226.635620,0.000383,0.000569,0.006616,960106.575390,1.332179,1232.922317


In [4]:
def initialize_feature_extractor():
    paramsFile = "CEM_extraction.yaml"
    extractor = featureextractor.RadiomicsFeatureExtractor(paramsFile, shape2D=True, force2D=True,
                                                               force2Ddimension=True, resampledPixelSpacing=None)
    extractor.addProvenance(False)
    extractor.disableAllFeatures()
    extractor.enableImageTypes(Original={})

    extractor.enableFeatureClassByName('firstorder', enabled=True)
    extractor.enableFeatureClassByName('shape2D', enabled=True)
    extractor.enableFeatureClassByName('glcm', enabled=True)
    extractor.enableFeatureClassByName('glrlm', enabled=True)
    extractor.enableFeatureClassByName('glszm', enabled=True)
    extractor.enableFeatureClassByName('gldm', enabled=True)
    extractor.enableFeatureClassByName('ngtdm', enabled=True)
    return extractor

In [5]:
def extract_slice(img, slice_no):
    size = list(img.GetSize())
    index = [0, 0, int(slice_no)]

    size[2] = 0  # extract 2D slice
    return sitk.Extract(img, size, index)

In [6]:
# testing stuff with paths
general_dir = Path(os.path.expanduser('~/project/xAI-in-NSCLC/NSCLC-Radiomics'))
path_records = []

for patient_dir in general_dir.iterdir():
    if not patient_dir.is_dir():
        continue

    scan_id = patient_dir.name
    
    for study_dir in patient_dir.iterdir():
        if not study_dir.is_dir():
            continue
        # skip hidden or system files like .DS_Store
        #if study_dir.name.startswith('.') or study_dir.name == '.DS_Store':
        #    continue

        #ct_series = None
        #seg_series = None

        for series_dir in study_dir.iterdir():
            if not series_dir.is_dir():
                continue

            if 'Segmentation' in series_dir.name and any(series_dir.glob('*.dcm')):
                seg_series = series_dir
                continue

            if any(series_dir.glob('*.dcm')) and len(list(series_dir.glob('*.dcm'))) >= 10:
                ct_series = series_dir

        if ct_series is not None and seg_series is not None:
            path_records.append({
                'scan_id': scan_id,
                'path_ct': ct_series,
                'path_mask':seg_series
            })
        else:
            print(f"Skipping {patient_dir.name}/{study_dir.name}: ct_series={ct_series is not None}, seg_series={seg_series is not None}")

path_df = pd.DataFrame(path_records, columns=['scan_id', 'path_ct', 'path_mask'])

In [8]:
path_df.head()

,scan_id,path_ct,path_mask
0,LUNG1-348,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...
1,LUNG1-339,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...
2,LUNG1-389,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...
3,LUNG1-155,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...
4,LUNG1-366,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...


In [9]:
path_df.to_csv('path_records.csv', index=False)

In [10]:
full_radiomics = full_radiomics[['PatientID', 'slice_no']]
grouped_min = full_radiomics.groupby(by='PatientID', as_index=False).min()
grouped_max = full_radiomics.groupby(by='PatientID', as_index=False).max()
min_max_df = grouped_min.merge(grouped_max, on='PatientID')

In [11]:
rows = []

for patient_id in min_max_df["PatientID"].unique():

    # Extract the row for this patient
    row = min_max_df.loc[min_max_df["PatientID"] == patient_id].iloc[0]

    # Convert to scalar ints
    min_slice = int(row["slice_no_x"])
    max_slice = int(row["slice_no_y"])

    # Get CT path
    ct_path = path_df.loc[path_df["scan_id"] == patient_id, "path_ct"].iloc[0]

    # Loop through DICOM files
    for f in os.listdir(ct_path):
        if f.endswith(".dcm"):
            slice_no = int(f.split('-')[1].split('.')[0])

            if min_slice <= slice_no <= max_slice:
                full_path = os.path.join(ct_path, f)
                rows.append({
                    "PatientID": patient_id,
                    "path": full_path
                })

result_df = pd.DataFrame(rows)

In [16]:
clinical_df = pd.read_csv(os.path.expanduser('~/project/xAI-in-NSCLC/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv'))

In [17]:
results_df = result_df.merge(clinical_df[['PatientID', 'Overall.Stage']], on='PatientID')
results_df.head()

,PatientID,path,Overall.Stage
0,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,IIIb
1,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,IIIb
2,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,IIIb
3,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,IIIb
4,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...,IIIb


In [18]:
result_df.head()

,PatientID,path
0,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...
1,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...
2,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...
3,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...
4,LUNG1-001,/home/coder/project/xAI-in-NSCLC/NSCLC-Radiomi...


In [19]:
results_df.to_csv('deeplearning_path.csv', index=False)

In [ ]:
#input_dcm_new = sorted(input_dcm.glob('*.dcm')) #makes list of all the paths in the directory and sorts
#seg_dcm_new = sorted(seg_dcm.glob('*.dcm')) #makes list of all the paths in the directory and sorts

records = []
# collect scan ids with mismatched geometry (seg vs ct)
mismatched_scans = []

seg_reader = dcmseg.SegmentReader()
extractor = initialize_feature_extractor()
ser_reader = sitk.ImageSeriesReader()

for _, row in path_df.iterrows():
    scan_id = row['scan_id']
    ct_path = row['path_ct']
    mask_path = row['path_mask']
    #for scan_id, ct_path, mask_path in zip(path_df['scan_id'], path_df['path_ct'], path_df['path_mask']):
    #seg_file = sorted(mask_path.glob('*.dcm'))

    #find correct file and read it as a dicom segmentation object
    seg = pydicom.dcmread(list(mask_path.glob('*.dcm'))[0])
    result_seg = seg_reader.read(seg)
    dcm_paths = sorted(ct_path.glob('*.dcm'))
    dcm_files = ser_reader.GetGDCMSeriesFileNames(str(ct_path))
    ser_reader.SetFileNames(dcm_files)
    sitk_dcms = ser_reader.Execute()


    # find segmentation from neoplasm label
    seg_infos = result_seg.segment_infos
    for seg_num, info in seg_infos.items():
        #could make this more efficient by stopping the loop if the correct item was found?
        if 'Neoplasm' not in info.get('SegmentLabel', ''):
            continue
        neo_seg_num = seg_num
        #print(f"Found neoplasm segment: {seg_num} with label {info['SegmentLabel']} in mask: {mask_path}")
        #Next step: execute pyradiomics per slice for each the dicom and the segmentation
        neoplasm_segment_img = result_seg.segment_image(neo_seg_num) # or should I do this..?
        #img_array_neoplasm = sitk.GetArrayFromImage(neoplasm_segment_img)
        #maybe sanity check that they have the same dimensions

        #need to cast the segmentation onto the same space as dicom image
        # this is important because radiomics will throw error because it thinks the segmentation is ever so slightly off due to data handling (by 0.0001 mm or so)
        fixed_seg = sitk.Cast(neoplasm_segment_img, sitk.sitkUInt8)
        # skip this scan if segmentation and CT have different sizes
        if fixed_seg.GetSize() != sitk_dcms.GetSize():
            print(f"Skipping {scan_id} due to size mismatch: seg={fixed_seg.GetSize()}, ct={sitk_dcms.GetSize()}")
            mismatched_scans.append(scan_id)
            continue
        fixed_seg.CopyInformation(sitk_dcms)
        for slice_no in range(sitk_dcms.GetSize()[2]):
            seg_slice = extract_slice(fixed_seg, slice_no)
            img_slice = extract_slice(sitk_dcms, slice_no)

            # Check if segmentation contains label 1
            if 1 not in sitk.GetArrayViewFromImage(seg_slice):
                continue

            features = extractor.execute(img_slice, seg_slice, label=1)
            record = {
                'patient_id': scan_id,
                'slice_no': slice_no,
            }
            record.update(features)
            records.append(record)

    #segment_sequence = result.available_segments
    #print(segment_sequence)

    #for segment in segment_sequence:
    #    print(seg.SegmentSequence[segment - 1].SegmentLabel)


In [ ]:
features_df = pd.DataFrame(records)
print(features_df.shape)
features_df.head()
print('Mismatched scans:', sorted(set(mismatched_scans)))

In [ ]:
features_df.to_csv('radiomics_features_per_slice.csv', index=False)

In [ ]:
# build a feature table with patient + slice identifier
records = []
for _, row in path_df.iterrows():
    scan_id = row['scan_id']
    ct_path = row['path_ct']
    mask_path = row['path_mask']

    seg = pydicom.dcmread(list(mask_path.glob('*.dcm'))[0])
    result_seg = seg_reader.read(seg)

    dcm_files = ser_reader.GetGDCMSeriesFileNames(str(ct_path))
    ser_reader.SetFileNames(dcm_files)
    sitk_dcms = ser_reader.Execute()

    seg_infos = result_seg.segment_infos
    for seg_num, info in seg_infos.items():
        if 'Neoplasm' not in info.get('SegmentLabel', ''):
            continue

        neo_seg_num = seg_num
        neoplasm_segment_img = result_seg.segment_image(neo_seg_num)
        fixed_seg = sitk.Cast(neoplasm_segment_img, sitk.sitkUInt8)
        # skip this scan if segmentation and CT have different sizes
        if fixed_seg.GetSize() != sitk_dcms.GetSize():
            print(f"Skipping {scan_id} due to size mismatch: seg={fixed_seg.GetSize()}, ct={sitk_dcms.GetSize()}")
            mismatched_scans.append(scan_id)
            continue
        fixed_seg.CopyInformation(sitk_dcms)

        seg_arr = sitk.GetArrayFromImage(fixed_seg)
        valid_slices = np.flatnonzero(np.any(seg_arr > 0, axis=(1, 2)))

        for slice_no in valid_slices:
            img_slice = extract_slice(sitk_dcms, slice_no)
            seg_slice = extract_slice(fixed_seg, slice_no)

            features = extractor.execute(img_slice, seg_slice)
            record = {
                'patient_id': f"{scan_id}_sl{slice_no}"
            }
            record.update(features)
            records.append(record)

features_df = pd.DataFrame(records)
#cols = ['patient_id'] + [c for c in features_df.columns if c not in {
#    'patient_id'}]
#features_df = features_df[cols]
print(features_df.shape)
features_df.head()

In [ ]:
dcm = pydicom.dcmread(input_dcm_new[70]) # takes first image
sitk_img = sitk.ReadImage(input_dcm_new[70]) # reads the dicom series as a sitk image


#You should preferably use the series reader!!!
series_reader = sitk.ImageSeriesReader()
files = series_reader.GetGDCMSeriesFileNames(input_dcm)
series_reader.SetFileNames(files)
sitk_imgs = series_reader.Execute() # reads the dicom series as a sitk image

In [ ]:
ct_arr = sitk.GetArrayFromImage(sitk_imgs)
print(ct_arr.shape)

In [ ]:
z = 80 # this is the slice number

arr = sitk.GetArrayFromImage(sitk_imgs)
slice = arr[z, :, :]
plt.imshow(slice, cmap='gray')
plt.show()

In [ ]:
#now do the same with the segmentation dicom...

dcm_seg = pydicom.dcmread(seg_dcm_new[0])

In [ ]:
reader = dcmseg.SegmentReader()
result = reader.read(dcm_seg)
print(result.available_segments)
print(dcm_seg.SegmentSequence[0].SegmentLabel) # you need to make sure that the neoplasm label is always in the same spot!
# in this case label 1 = neoplasm, but is this always tje case?
#Are there different labels than neoplasm that correspond to the subtype of the tumour?

In [ ]:
neoplasm_segment = result.segment_data(1) # this is the neoplasm segment, but is this always the case?
print(neoplasm_segment.shape)
neoplasm_segment_img = result.segment_image(1) # or should I do this..?
img_array_neoplasm = sitk.GetArrayFromImage(neoplasm_segment_img)
print(img_array_neoplasm.shape)
print(len(input_dcm_new))

plt.imshow(slice, cmap='gray') # overlay the original image with the segmentation mask
plt.imshow(img_array_neoplasm[80], cmap='gray', alpha=0.2) # overlay the segmentation mask with the original image
plt.show()
#something is wrong with the segmentation mask, it doesn't fit the original ct

In [ ]:
print(dcm.SeriesInstanceUID)
print(dcm_seg.ReferencedSeriesSequence[0].SeriesInstanceUID)
print(dcm.GantryDetectorTilt)
print(dcm_seg.SharedFunctionalGroupsSequence[0]
      .PixelMeasuresSequence[0].PixelSpacing, dcm.PixelSpacing)
print(dcm_seg.SharedFunctionalGroupsSequence[0]
      .PixelMeasuresSequence[0].SliceThickness, dcm.SliceThickness)


In [ ]:
def initialize_feature_extractor():
    paramsFile = "CEM_extraction.yaml"
    extractor = featureextractor.RadiomicsFeatureExtractor(paramsFile, shape2D=True, force2D=True,
                                                               force2Ddimension=True, resampledPixelSpacing=None)
    extractor.addProvenance(False)
    extractor.disableAllFeatures()
    extractor.enableImageTypes(Original={})

    extractor.enableFeatureClassByName('firstorder', enabled=True)
    extractor.enableFeatureClassByName('shape2D', enabled=True)
    extractor.enableFeatureClassByName('glcm', enabled=True)
    extractor.enableFeatureClassByName('glrlm', enabled=True)
    extractor.enableFeatureClassByName('glszm', enabled=True)
    extractor.enableFeatureClassByName('gldm', enabled=True)
    extractor.enableFeatureClassByName('ngtdm', enabled=True)
    return extractor

In [ ]:
extractor = initialize_feature_extractor()
fixed_seg = sitk.Cast(neoplasm_segment_img, sitk.sitkUInt8)
fixed_seg.CopyInformation(sitk_imgs)

In [ ]:
def extract_slice(img, slice_no):
    size = list(img.GetSize())
    index = [0, 0, slice_no]

    size[2] = 0  # extract 2D slice
    return sitk.Extract(img, size, index)

In [ ]:
for slice_no in range(80, 85):
    img_slice = extract_slice(sitk_imgs, slice_no)
    seg_slice = extract_slice(fixed_seg, slice_no)

    # Check if segmentation contains label 1
    if 1 not in sitk.GetArrayViewFromImage(seg_slice):
        continue

    features = extractor.execute(img_slice, seg_slice, label=1)
    print(f" features:", features)

In [ ]:
features

In [ ]:

result_extraction = extractor.execute(sitk_imgs, neoplasm_segment_img)

In [ ]:
input_dcm = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/0.000000-NA-82046'))
seg_dcm = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/300.000000-Segmentation-9.554'))
output_dicom_dir = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/nii_output'))
series_IDs = sitk.ImageSeriesReader.GetGDCMSeriesIDs(input_dcm)

itk_images = []
for i in range(0,len(series_IDs)):
  series_file_names = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(input_dcm, series_IDs[i])
  series_reader = sitk.ImageSeriesReader()
  series_reader.SetFileNames(series_file_names)
  series_reader.MetaDataDictionaryArrayUpdateOn()
  series_reader.LoadPrivateTagsOn()
  image_dicom = series_reader.Execute()
  
  itk_images.append(image_dicom)
  sitk.WriteImage(image_dicom, os.path.join(output_dicom_dir, series_IDs[i] + ".nii.gz"))

In [ ]:
def generate_features_table(df, extractor,inference_usage=False):
    # warning: this function can take a long time to run
    # extract low energy features
    featureVector_low_energy = extractor.execute(list(df["path_low_energy"])[0], list(df["path_mask"])[0])
    temp_dataset = pd.Series(featureVector_low_energy)
    feature_df_low_energy = pd.DataFrame([temp_dataset], columns=list(featureVector_low_energy.keys()),
                                         index=[list(df["path_mask"])[0]])
    for i, temp_mask in tqdm.tqdm(enumerate(list(df["path_mask"])[1:])):
        featureVector_low_energy = extractor.execute(list(df["path_low_energy"])[i + 1], temp_mask)
        temp_dataset = pd.Series(featureVector_low_energy)
        feature_df_low_energy.loc[temp_mask] = temp_dataset.values
    feature_df_low_energy.columns = feature_df_low_energy.columns + "_low_energy"